# 第12回 強化学習：値とコードを一段ずつ読む

環境で得た経験を、Q表のどの1マスへ反映するかを追います。報酬・目標値・差・更新後を分けて読みます。

**開始：** 既存venvのカーネルを選び、初回は変更せず「カーネル再起動→すべて実行」。次に一節ずつ、予想してからShift+Enterで実行します。同梱Pythonのimportは不要で、外部通信も行いません。

**読み方：** 関数の定義は計算方法に名前を付けるセルです。続く短い実行セルが入力を渡します。表・図の表示コードは折り畳み指定です。表示は丸めますが計算には元の値を使います。

**完了の目印：** 手計算例の目標6.4・差4.4・新Q4.2、実際の行動履歴とQ表の更新位置が対応する。


In [ ]:
import sys
print('使用中のPython:',sys.executable)
print('版:',sys.version.split()[0])


In [ ]:
import argparse


In [ ]:
from copy import deepcopy


In [ ]:
import json


In [ ]:
import math


In [ ]:
BASE = {'alpha': 0.5, 'gamma': 0.9, 'epsilon': 0.3,
        'episodes': 100, 'seed': 65535, 'delayed_reward': 10.0}


In [ ]:
CHANGES = {'alpha': 0.1, 'gamma': 0.2, 'epsilon': 0.0,
           'episodes': 10, 'delayed_reward': 2.0, 'seed': 1}


In [ ]:
STATES = (0, 2)


In [ ]:
ACTIONS = ('left', 'right')


### 図表の準備（変更不要）

表示用の補助コードです。HTML/SVGを覚える必要はありません。折り畳み表示はNotebook環境によって異なります。


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def learning_format(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (list, tuple)):
        return '['+', '.join(learning_format(item) for item in value)+']'
    if isinstance(value, dict):
        return '{'+', '.join(str(key)+': '+learning_format(item) for key,item in value.items())+'}'
    return str(value)


In [ ]:
def learning_table(title, columns, rows):
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''
    for row in rows:
        body += '<tr>'+''.join('<td>'+escape(learning_format(x))+'</td>' for x in row)+'</tr>'
    return '<div class="scroll"><table><caption>'+escape(title)+'</caption><tr>'+header+'</tr>'+body+'</table></div>'


In [ ]:
def learning_grid(title, matrix):
    rows = [(i,*row) for i,row in enumerate(matrix)]
    return learning_table(title,['行↓ 列→',*range(len(matrix[0]))],rows)


In [ ]:
def learning_plot(title, points):
    # 実計算の値だけを座標へ変換。点の間の値は補間して表示しない。
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    xmin,xmax,ymin,ymax = min(xs),max(xs),min(0,min(ys)),max(ys)
    sx = lambda x: 100+(x-xmin)/max(1e-12,xmax-xmin)*500
    sy = lambda y: 240-(y-ymin)/max(1e-12,ymax-ymin)*190
    marks = ''
    for value in (ymin,(ymin+ymax)/2,ymax):
        marks += f'<path d="M100 {sy(value)} H600" stroke="#aaa"/><text x="0" y="{sy(value)+5}">{value:.4g}</text>'
    for x,y in points:
        marks += f'<circle cx="{sx(x)}" cy="{sy(y)}" r="4" fill="#155e75"><title>{x}: {y}</title></circle>'
    marks += f'<text x="100" y="275">{xmin}</text><text x="580" y="275">{xmax}</text>'
    return '<svg viewBox="0 0 700 310" role="img" aria-label="'+escape(title)+'"><text x="100" y="22">'+escape(title)+'</text>'+marks+'</svg>'


In [ ]:
def learning_show(report, reset=None):
    style = '<style>.learning{font:17px/1.65 sans-serif;color:#172b42;background:white;padding:12px}.learning .scroll{overflow-x:auto}.learning table{border-collapse:collapse;margin:14px 0}.learning td,.learning th{border:1px solid #8394a5;padding:6px 12px;white-space:normal}.learning caption{text-align:left;font-weight:bold}.learning svg{max-width:100%;height:auto}.learning .boards{display:flex;gap:20px;flex-wrap:wrap}.learning .result{border-left:4px solid #155e75;padding:10px}</style>'
    content = '<h3>'+escape(report['title'])+'</h3><p>入力：'+escape(learning_format(report['input']))+'</p>'
    content += learning_table('使う値 → 操作 → 結果',report['columns'],report['rows'])
    for title,columns,rows in report.get('tables',[]):
        content += '<details><summary>詳しい計算：'+escape(title)+'</summary>'+learning_table(title,columns,rows)+'</details>'
    content += '<div class="boards">'+''.join(learning_grid(title,values) for title,values in report.get('matrices',[]))+'</div>'
    for title,points in report.get('plots',[]):
        content += learning_plot(title,points)
    for label in ('result','comparison','explanation','reset'):
        name={'result':'結果','comparison':'一つ変えた比較','explanation':'読み取り方','reset':'戻す'}[label]
        content += '<p class="result">'+name+'：'+escape(reset if label=='reset' and reset is not None else learning_format(report[label]))+'</p>'
    display(HTML(style+'<div class="learning">'+content+'</div>'))


## 1. 環境の状態遷移

**今いる段階：** 状態→行動→報酬→次状態

**予想：** 右の直後が報酬0なら価値も0か

**操作：** example_rewardを10から2へ変更し、同じセルを実行します。

**結果の目印：** 右経路のγ0.9の収益は9、遅延報酬2なら1.8

**戻す：** example_reward=10に戻して再実行します。

**説明する：** 目先と将来の報酬を分ける理由


### 計算の定義：`number`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def number(x, low, high, name='値'):
    if isinstance(x, bool) or not isinstance(x, (int, float)) or not math.isfinite(x) or not low <= x <= high:
        raise ValueError(f'{name}は{low}〜{high}の有限数です')
    return x


### 計算の定義：`integer`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def integer(x, low, high, name='値'):
    number(x, low, high, name)
    if not isinstance(x, int):
        raise ValueError(f'{name}は整数です')
    return x


### 計算の定義：`state_index`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def state_index(state):
    integer(state, 0, 4, '状態')
    if state not in STATES:
        raise ValueError('終端では行動できません')
    return STATES.index(state)


### 計算の定義：`transition`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def transition(state, action, delayed_reward=10):
    state_index(state)
    if action not in ACTIONS:
        raise ValueError('行動はleftまたはrightです')
    number(delayed_reward, -100, 100, '遅延報酬')
    if state == 0:
        return (1, 4.0, True) if action == 'left' else (2, 0.0, False)
    return (3, 0.0, True) if action == 'left' else (4, delayed_reward, True)


### 計算の定義：`path_return`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def path_return(rewards, gamma):
    number(gamma, 0, 1, '割引率')
    if not isinstance(rewards, list) or not 1 <= len(rewards) <= 2:
        raise ValueError('報酬列は1〜2個です')
    for r in rewards:
        number(r, -100, 100, '報酬')
    return {'sum': sum(rewards), 'discounted': sum(gamma**i * r for i, r in enumerate(rewards))}


### 計算の定義：`lesson_environment`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_environment(delayed_reward=10):
    rows = []
    for state in STATES:
        for action in ACTIONS:
            next_state,reward,terminal = transition(state,action,delayed_reward)
            rows.append((state,action,reward,next_state,terminal))
    return {'title':'環境：状態→行動→報酬→次状態','question':'右を選んだ直後の報酬が0でも、その先へ進む価値はあるか',
            'input':f'行動できる状態は0と2。終端は1・3・4。遅延報酬={delayed_reward}',
            'columns':['現在状態','行動','今の報酬','次状態','終端か'],'rows':rows,
            'tables':[('二つの経路を比較',['経路','報酬列','合計','γ=0.9の割引収益'],
                      [('0→1','[4]',4,path_return([4],.9)['discounted']),('0→2→4',str([0,delayed_reward]),delayed_reward,path_return([0,delayed_reward],.9)['discounted'])])],
            'result':'左は1回で終了、右は状態2でもう一度行動する。',
            'comparison':'遅延報酬を10→2にすると、右経路の割引収益は9→1.8。',
            'explanation':'報酬は環境から受け取り、Q値は経験から更新する。終端では次の行動がない。',
            'reset':'delayed_reward=10で再実行。'}


In [ ]:
example_reward = 10
program_observation_environment = lesson_environment(example_reward)
learning_show(program_observation_environment, reset='example_reward=10に戻して再実行します。')


## 2. Q更新式を数値で追う

**今いる段階：** 目標→差→新Q

**予想：** 終端かどうかで目標は変わるか

**操作：** example_alphaを0.5から0へ変更し、同じセルを実行します。

**結果の目印：** 非終端6.4、終端1

**戻す：** example_alpha=0.5に戻して再実行します。

**説明する：** αとγの役割の違い


### 計算の定義：`update_value`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def update_value(old, reward, next_max, terminal, alpha, gamma):
    for name, value in [('旧Q値', old), ('報酬', reward), ('次の最大Q値', next_max)]:
        number(value, -100, 100, name)
    number(alpha, 0, 1, '学習率')
    number(gamma, 0, 1, '割引率')
    if not isinstance(terminal, bool):
        raise ValueError('終端条件はboolです')
    target = reward if terminal else reward + gamma * next_max
    return {'old': old, 'reward': reward, 'next_max': next_max, 'terminal': terminal,
            'target': target, 'error': target - old, 'new': old + alpha * (target - old)}


### 計算の定義：`lesson_update`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_update(alpha=0.5):
    result = update_value(2,1,6,False,alpha,.9)
    terminal = update_value(2,1,6,True,alpha,.9)
    rows = [('非終端',result['old'],result['reward'],result['next_max'],.9,result['target'],result['error'],alpha,result['new']),
            ('終端',terminal['old'],terminal['reward'],'使わない',.9,terminal['target'],terminal['error'],alpha,terminal['new'])]
    return {'title':'Q更新：目標値→差→更新後','question':'同じ数値でも次が終端なら目標はどう変わるか',
            'input':'式を確かめる手計算例：旧Q2、報酬1、次の最大Q6、γ0.9。実環境の報酬とは区別。',
            'columns':['条件','旧Q','報酬','次の最大Q','γ','目標','差','α','新Q'],'rows':rows,
            'result':f"2＋{alpha}×({result['target']}−2)={result['new']}",
            'comparison':f"α=0では{update_value(2,1,6,False,0,.9)['new']}、α=1では{update_value(2,1,6,False,1,.9)['new']}",
            'explanation':'非終端の目標は報酬＋γ×次の最大Q。終端なら報酬だけ。αは目標へ近づく割合。',
            'reset':'alpha=0.5で再実行。'}


In [ ]:
example_alpha = 0.5
program_observation_update = lesson_update(example_alpha)
learning_show(program_observation_update, reset='example_alpha=0.5に戻して再実行します。')


## 3. 経験とQ表のマスを対応させる

**今いる段階：** 経験→更新前Q→更新後Q

**予想：** 1回の行動で何マス変えるか

**操作：** example_stepsを6から12へ変更し、同じセルを実行します。

**結果の目印：** 各経験の状態行・行動列だけ変わる

**戻す：** example_steps=6に戻して再実行します。

**説明する：** 行動回数とエピソード数の違い


### 計算の定義：`settings`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def settings(config=None):
    result = dict(BASE)
    if config is not None:
        if not isinstance(config, dict) or set(config) - set(BASE):
            raise ValueError('設定名を確認してください')
        result.update(config)
    for k in ('alpha', 'gamma', 'epsilon'):
        number(result[k], 0, 1, k)
    integer(result['episodes'], 0, 200, 'エピソード数')
    integer(result['seed'], 0, 2**32 - 1, 'シード')
    number(result['delayed_reward'], -100, 100, '遅延報酬')
    return result


### 計算の定義：`RNG`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
class RNG:
    def __init__(self, seed):
        self.state = integer(seed, 0, 2**32 - 1, 'シード')

    def random(self):
        self.state = (1664525 * self.state + 1013904223) % 2**32
        return self.state / 2**32


### 計算の定義：`greedy`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def greedy(row):
    if not isinstance(row, list) or len(row) != 2:
        raise ValueError('左右のQ値を指定してください')
    for value in row:
        number(value, -100, 100, 'Q値')
    return 0 if row[0] >= row[1] else 1


### 計算の定義：`probabilities`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def probabilities(row, epsilon):
    number(epsilon, 0, 1, '探索率')
    best = greedy(row)
    return [epsilon / 2 + (1 - epsilon if a == best else 0) for a in range(2)]


### 計算の定義：`select_action`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def select_action(row, epsilon, rng):
    probabilities(row, epsilon)  # 乱数消費前に検証する。
    coin, choice = rng.random(), rng.random()  # 活用時にも必ず2個消費する。
    explore = coin < epsilon
    a = int(choice * 2) if explore else greedy(row)
    return {'action': ACTIONS[a], 'explore': explore, 'draws': [coin, choice]}


### 計算の定義：`qtable`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def qtable(q):
    if not isinstance(q, list) or len(q) != 2:
        raise ValueError('Q表は状態0・2の2行です')
    for row in q:
        if not isinstance(row, list) or len(row) != 2:
            raise ValueError('各行は左・右の2列です')
        for value in row:
            number(value, -100, 100, 'Q値')
    return q


### 計算の定義：`learn`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def learn(q, state, action, alpha=0.5, gamma=0.9, delayed_reward=10):
    qtable(q)
    i = state_index(state)
    nxt, reward, terminal = transition(state, action, delayed_reward)
    a = ACTIONS.index(action)
    future = 0.0 if terminal else max(q[state_index(nxt)])
    result = update_value(q[i][a], reward, future, terminal, alpha, gamma)
    changed = deepcopy(q)
    changed[i][a] = result['new']
    return changed, {'state': state, 'action': action, 'next_state': nxt, **result}


### 計算の定義：`evaluate`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def evaluate(q, gamma=0.9, delayed_reward=10):
    qtable(q)
    number(gamma, 0, 1, '割引率')
    number(delayed_reward, -100, 100, '遅延報酬')
    state, path, actions, rewards = 0, [0], [], []
    while state in STATES:
        a = ACTIONS[greedy(q[state_index(state)])]
        state, reward, _ = transition(state, a, delayed_reward)
        path.append(state)
        actions.append(a)
        rewards.append(reward)
    return {'path': path, 'actions': actions, 'rewards': rewards, **path_return(rewards, gamma)}


### 計算の定義：`Trainer`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
class Trainer:
    def __init__(self, config=None):
        self.config = settings(config)
        self.q = [[0.0, 0.0], [0.0, 0.0]]
        self.rng = RNG(self.config['seed'])
        self.state, self.completed = 0, 0
        self.history = []

    def step(self):
        if self.completed >= self.config['episodes']:
            return None
        if self.state not in STATES:
            self.state = 0
        choice = select_action(self.q[state_index(self.state)], self.config['epsilon'], self.rng)
        self.q, record = learn(self.q, self.state, choice['action'], self.config['alpha'],
                               self.config['gamma'], self.config['delayed_reward'])
        record.update(episode=self.completed + 1, explore=choice['explore'], draws=choice['draws'])
        self.history.append(record)
        self.state = record['next_state']
        if record['terminal']:
            self.completed += 1
        return deepcopy(record)

    def episode(self):
        before = self.completed
        while self.completed == before and self.step() is not None:
            pass
        return self.result()

    def finish(self):
        while self.step() is not None:
            pass
        return self.result()

    def result(self):
        return {'config': dict(self.config), 'q': deepcopy(self.q), 'completed': self.completed,
                'state': self.state, 'rng_state': self.rng.state, 'history': deepcopy(self.history),
                'evaluation': evaluate(self.q, self.config['gamma'], self.config['delayed_reward'])}


### 計算の定義：`lesson_training`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_training(steps=6):
    trainer = Trainer()
    rows, tables = [], []
    for index in range(steps):
        before = deepcopy(trainer.q)
        record = trainer.step()
        if record is None:
            break
        rows.append((index+1,record['state'],record['action'],record['reward'],record['next_state'],record['terminal'],record['old'],record['target'],record['error'],record['new']))
        tables.append((f'経験{index+1}で変えたQ表',['状態','更新前の左','更新前の右','更新後の左','更新後の右'],
                       [(state,*before[i],*trainer.q[i]) for i,state in enumerate(STATES)]))
    return {'title':'実際の経験からQ表の1マスを更新する','question':'一つの行動でQ表の全部を書き換えるか',
            'input':f'初期Qは全0、固定seed、{steps}行動。行動数とエピソード数を区別。',
            'columns':['行動番号','状態','行動','報酬','次状態','終端','旧Q','目標','差','新Q'],'rows':rows,'tables':tables,
            'result':f'完了エピソード={trainer.completed}、Q表={trainer.q}',
            'comparison':'stepsを6→12にすると、同じ初期状態から12行動分を見直す。前の6行動の続きではない。',
            'explanation':'各経験の状態行と行動列の1マスだけを更新する。目標には更新前のQ表を使う。終端を経験すると1エピソードが完了する。',
            'reset':'steps=6で再実行。Trainerを新しく作り、Q表と乱数状態を戻す。'}


In [ ]:
example_steps = 6
program_observation_training = lesson_training(example_steps)
learning_show(program_observation_training, reset='example_steps=6に戻して再実行します。')


## 4. 探索と活用の選択

**今いる段階：** 乱数→選択方法→行動

**予想：** 探索は必ず右か

**操作：** example_epsilonを0.3から0へ変更し、同じセルを実行します。

**結果の目印：** ε0では高いQの左、ε1では左右同確率

**戻す：** example_epsilon=0.3に戻して再実行します。

**説明する：** 探索が良い行動を選ぶ場合もある理由


### 計算の定義：`lesson_exploration`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_exploration(epsilon=0.3):
    row = [4.,0.]
    rng = RNG(BASE['seed'])
    rows = []
    for index in range(8):
        choice = select_action(row,epsilon,rng)
        rows.append((index+1,*choice['draws'],choice['explore'],choice['action']))
    return {'title':'探索と活用：どの行動を選ぶか','question':'探索したとき必ず悪い方の行動になるか',
            'input':f'Qの左4・右0、探索率ε={epsilon}。Q表はこの比較では固定。',
            'columns':['回','探索判定の乱数','探索時の行動乱数','探索か','選んだ行動'],'rows':rows,
            'result':f'左・右の選択確率={probabilities(row,epsilon)}',
            'comparison':f'ε=0の確率={probabilities(row,0)}、ε=1の確率={probabilities(row,1)}',
            'explanation':'探索では左右から選ぶため、良い方を選ぶ場合もある。εは行動選択、αは更新量、γは将来価値の重み。',
            'reset':'epsilon=0.3で再実行。同じseedから8回選ぶ。'}


In [ ]:
example_epsilon = 0.3
program_observation_exploration = lesson_exploration(example_epsilon)
learning_show(program_observation_exploration, reset='example_epsilon=0.3に戻して再実行します。')


## 5. 更新せず方策を評価

**今いる段階：** 学習Q→固定方策→収益

**予想：** 遠い報酬の重みを下げると経路は変わるか

**操作：** example_gammaを0.9から0.2へ変更し、同じセルを実行します。

**結果の目印：** 評価経路と割引収益が表示される

**戻す：** example_gamma=0.9に戻して再実行します。

**説明する：** 参照Qと学習したQを区別


### 計算の定義：`train`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def train(config=None):
    return Trainer(config).finish()


### 計算の定義：`optimal_q`

既知環境の参照値。学習処理からは呼ばない。


In [ ]:
def optimal_q(gamma=0.9, delayed_reward=10):
    """既知環境の参照値。学習処理からは呼ばない。"""
    number(gamma, 0, 1, '割引率')
    number(delayed_reward, -100, 100, '遅延報酬')
    return [[4.0, gamma * max(0, delayed_reward)], [0.0, delayed_reward]]


### 計算の定義：`lesson_evaluation`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_evaluation(gamma=0.9):
    result = train({'gamma':gamma})
    evaluation = result['evaluation']
    rows = [(i,state,evaluation['actions'][i],evaluation['rewards'][i],gamma**i,evaluation['rewards'][i]*gamma**i) for i,state in enumerate(evaluation['path'][:-1])]
    return {'title':'学習した方策を、更新せず評価する','question':'報酬の合計と割引収益は同じか',
            'input':f'100エピソード後のQ表、γ={gamma}、評価時はε=0・Q更新なし。',
            'columns':['時刻','状態','行動','報酬','割引係数','割引した報酬'],'rows':rows,
            'tables':[('Q表：学習値と既知環境の参照値は別',['状態','学習した左','学習した右','参照の左','参照の右'],[(state,*result['q'][i],*optimal_q(gamma)[i]) for i,state in enumerate(STATES)])],
            'result':evaluation,'comparison':'γを0.9→0.2にすると、遠い報酬の重みが小さくなる。',
            'explanation':'学習中の探索経路を、そのまま評価経路と呼ばない。既知環境の最適Qを学習で得た値と混同しない。',
            'reset':'gamma=0.9で再実行。初期Q・seedから再学習して評価する。'}


In [ ]:
example_gamma = 0.9
program_observation_evaluation = lesson_evaluation(example_gamma)
learning_show(program_observation_evaluation, reset='example_gamma=0.9に戻して再実行します。')


## 基準結果と既存の変更実験

ここまでの図表を、従来の基準出力と照合します。以下の実験も入力のコピーを使います。定義を変更した場合は再起動して全セル実行してください。


### 計算の定義：`summary`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def summary(result):
    return {k: deepcopy(result[k]) for k in ('config', 'q', 'completed', 'rng_state', 'evaluation')}


### 計算の定義：`round6`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def round6(value):
    if isinstance(value, float):
        return round(value, 6)
    if isinstance(value, list):
        return [round6(v) for v in value]
    if isinstance(value, dict):
        return {k: round6(v) for k, v in value.items()}
    return value


### 計算の定義：`demo`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def demo():
    base = train()
    return round6({'baseline': summary(base), 'first_updates': base['history'][:6],
                   'optimal_reference': optimal_q(),
                   'changes': {k: summary(train({**BASE, k: v})) for k, v in CHANGES.items()}})


## 定義

状態0・2の左と右に対応する2×2のQ表を使います。終端1・3・4では行動しません。計算途中は丸めません。


## 基準結果を保持する

100エピソード、α=0.5、γ=0.9、ε=0.3。予想を先に書いてください。評価はQ表を固定し、探索・更新なしで行います。


In [ ]:
baseline_config = dict(BASE)
baseline_result = demo()
baseline_run = train(baseline_config)
print(json.dumps(baseline_result["baseline"], ensure_ascii=False, indent=2))


基準の評価経路は0→2→4、報酬合計10、割引収益9。Q表の9・10は6桁表示であり、有限回で厳密な最適値に到達したという意味ではありません。


## 変更：alpha

問い・予想：学習率を小さくすると更新幅はどうなるか。終端報酬10の最初の更新は1になる。

基準のコピーから一項目だけ変えます。Q表、評価経路、割引収益を比較し、違いを説明してください。


In [ ]:
trial_alpha_config = dict(baseline_config)
trial_alpha_config["alpha"] = 0.1
trial_alpha = train(trial_alpha_config)
print(json.dumps(round6(summary(trial_alpha)), ensure_ascii=False, indent=2))


## 変更：gamma

問い・予想：遅い報酬の評価はどうなるか。参照値は状態0の左4・右2。

基準のコピーから一項目だけ変えます。Q表、評価経路、割引収益を比較し、違いを説明してください。


In [ ]:
trial_gamma_config = dict(baseline_config)
trial_gamma_config["gamma"] = 0.2
trial_gamma = train(trial_gamma_config)
print(json.dumps(round6(summary(trial_gamma)), ensure_ascii=False, indent=2))


## 変更：epsilon

問い・予想：探索しない場合は右を発見できるか。同点は左優先なので経路0→1を選び続ける。

基準のコピーから一項目だけ変えます。Q表、評価経路、割引収益を比較し、違いを説明してください。


In [ ]:
trial_epsilon_config = dict(baseline_config)
trial_epsilon_config["epsilon"] = 0
trial_epsilon = train(trial_epsilon_config)
print(json.dumps(round6(summary(trial_epsilon)), ensure_ascii=False, indent=2))


## 変更：episodes

問い・予想：短い学習で十分か。基準より履歴が短くなるが、性能向上の保証はない。

基準のコピーから一項目だけ変えます。Q表、評価経路、割引収益を比較し、違いを説明してください。


In [ ]:
trial_episodes_config = dict(baseline_config)
trial_episodes_config["episodes"] = 10
trial_episodes = train(trial_episodes_config)
print(json.dumps(round6(summary(trial_episodes)), ensure_ascii=False, indent=2))


## 変更：delayed_reward

問い・予想：遠い終端の報酬が2ならどちらが有利か。参照値の右は1.8、左は4。

基準のコピーから一項目だけ変えます。Q表、評価経路、割引収益を比較し、違いを説明してください。


In [ ]:
trial_delayed_reward_config = dict(baseline_config)
trial_delayed_reward_config["delayed_reward"] = 2.0
trial_delayed_reward = train(trial_delayed_reward_config)
print(json.dumps(round6(summary(trial_delayed_reward)), ensure_ascii=False, indent=2))


## 変更：seed

問い・予想：同じ設定でも探索の順序は変わるか。行動履歴を比較し、最終経路が同じでも過程が同じとは限らない。

基準のコピーから一項目だけ変えます。Q表、評価経路、割引収益を比較し、違いを説明してください。


In [ ]:
trial_seed_config = dict(baseline_config)
trial_seed_config["seed"] = 1
trial_seed = train(trial_seed_config)
print(json.dumps(round6(summary(trial_seed)), ensure_ascii=False, indent=2))


## 基準を確認・元に戻す

下の確認が成功すれば基準は保持されています。変更したコピーを捨て、カーネル再起動→全セル実行で復帰できます。未定義名は途中セルだけ実行していないか、ValueErrorは設定範囲を確認してください。


In [ ]:
assert BASE == baseline_config
assert baseline_result == demo()
assert baseline_run == train(baseline_config)
print("基準入力・結果を保持しました")


## 振り返りと復帰

入力・行った処理・結果・変更前との差を、自分の言葉で説明してください。観察セルは毎回基準のコピーから計算するため、同じセルを再実行しても前の実験の続きにはなりません。NameErrorや実行順が不明な場合はカーネル再起動→全セル実行。元の配布ファイルを残し、変更版は別名保存します。

Python版は`python agent_qlearning.py --explain all`で同じ計算表を表示します。各テーマは`--explain environment`のように指定できます。従来のコマンドと固定期待値も維持しています。
